# HCDE 530 — A5: Pandas Analysis

### Instructions
In class you used pandas to answer five questions about a dataset. For A5 you are doing that on your own data — the dataset you chose for Mini Project 1 — and documenting what you found.

This assignment is closely connected to MP1. The analysis you do here is the foundation for the notebook you will build in Week 6.

### Introduction
League of Legends is an online multiplayer game where two teams of 5 battle against each other in what I can only describe as a glorified capture the flag game. Each team must destroy the other team's Nexus in their base to win the game. There are 5 unique roles on each team, corresponding to the lane or area they're typically supposed to play in: Top, Middle, Bottom, Support, and Jungle. In competitive play, players are ranked from Iron to Challenger based on their performance. 

As a Support main that hovers around the Bronze level, I was curious in using the Riot Developer API to answer the following questions: 
1. Who is the most played Support champion in Bronze IV Solo 5x5 Ranked lobbies? What about across all ranks? 
2. What does the distribution of the most popular Support player across every rank look like? 
3. What is the most common Bottom Lane/Support pairing in Bronze-level lobbies? What about across all ranks? 

A previous version of these questions included one about the pick rate of cosmetic skins, but because that information isn't available through the Riot Developer API, I've changed that question to one that can actually be answered by the API. 

---
### 0. Setting up Pandas

In [3]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print("pandas version:", pd.__version__)

pandas version: 3.0.2


---
### 1. Importing up the API

To begin answering my data analysis questions, I need to set up endpoints for 3 different APIs, each dealing with a different type of player information. This includes one to fetch summoner ID data (where I'll query for Player Universally Unique IDs (PUUID)), one to extract game match information based on those PUUIDs, and finally an API to extract champion picks per match. 

##### Player Universally Unique IDs

In [ ]:
import json
import os
import time
import urllib.error
import urllib.request
from pathlib import Path

#function to load .env file so that this code can read the API key on it 
def _load_env(path: Path) -> None:
    """Load KEY=VALUE pairs from a .env file into os.environ (does not override existing vars).""" 
    if not path.is_file(): #if the .env file doesn't exist, return None
        return
    for line in path.read_text().splitlines():# for each line in the .env file
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, val = line.partition("=")
        key, val = key.strip(), val.strip().strip('"').strip("'")
        if key:
            os.environ.setdefault(key, val)


# Resolve Week 5/.env whether the kernel cwd is the repo root or the Week 5 folder
for _env in (Path(".env"), Path("Week 5") / ".env"):
    _load_env(_env)

api_key = os.environ.get("RIOT_API_KEY", "").strip() #the api key I stored in .env, will be regenerated every 24 hours
if not api_key:
    raise ValueError("RIOT_API_KEY missing: add it to Week 5/.env or set it in the environment.")

base = "https://na1.api.riotgames.com/lol/league/v4/entries/RANKED_SOLO_5x5/BRONZE/IV" #endpoint for summoner ID data, need to query for PUUIDs later
# Riot sits behind Cloudflare; default User-Agent "Python-urllib/..." often gets HTTP 403 + "error code: 1010".
headers = {
    "X-Riot-Token": api_key,
    "User-Agent": "HCDE530-A5/1.0 (pandas coursework; contact instructor)",
}

all_entries: list = [] #list to store all the data fetched from the API
for page in range(1, 326):  # pages 1 … 325 #calculated based on API request limit from Riot, but actually only fetched 258 because page 259 had no data (see break function below)
    url = f"{base}?page={page}"
    req = urllib.request.Request(url, headers=headers)
    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            data = json.load(resp)
    except urllib.error.HTTPError as e:
        body = e.read().decode(errors="replace")
        raise RuntimeError(f"HTTP {e.code} {e.reason}: {body[:500]}") from e

    if not data:
        break
    all_entries.extend(data)
    print(f"page {page}", flush=True) #print the page number to track progress as the loop runs
    time.sleep(2.0) #wait 2 seconds between requests to avoid overwhelming the server

summoner_league_json = all_entries #assign the fetched data to a variable to convert JSON into dataframe later 
print(len(summoner_league_json), "total entries across fetched pages") #total data fetched across each request aka page 
if summoner_league_json:
    print("example keys:", list(summoner_league_json[0].keys())) #fields in the JSON data fetched from the API

page 1
page 2
page 3
page 4
page 5
page 6
page 7
page 8
page 9
page 10
page 11
page 12
page 13
page 14
page 15
page 16
page 17
page 18
page 19
page 20
page 21
page 22
page 23
page 24
page 25
page 26
page 27
page 28
page 29
page 30
page 31
page 32
page 33
page 34
page 35
page 36
page 37
page 38
page 39
page 40
page 41
page 42
page 43
page 44
page 45
page 46
page 47
page 48
page 49
page 50
page 51
page 52
page 53
page 54
page 55
page 56
page 57
page 58
page 59
page 60
page 61
page 62
page 63
page 64
page 65
page 66
page 67
page 68
page 69
page 70
page 71
page 72
page 73
page 74
page 75
page 76
page 77
page 78
page 79
page 80
page 81
page 82
page 83
page 84
page 85
page 86
page 87
page 88
page 89
page 90
page 91
page 92
page 93
page 94
page 95
page 96
page 97
page 98
page 99
page 100
page 101
page 102
page 103
page 104
page 105
page 106
page 107
page 108
page 109
page 110
page 111
page 112
page 113
page 114
page 115
page 116
page 117
page 118
page 119
page 120
page 121
page 122
page 123
p

---
### 2. Turning JSON data into a DataFrame

In [ ]:
# List of dicts from section 1a — after the fetch cell runs, `summoner_league_json` and `all_entries` point to the same list.
if "summoner_league_json" in globals():
    rows = summoner_league_json
elif "all_entries" in globals():
    rows = all_entries
else:
    raise RuntimeError("Run the API cell in section 1a first so summoner_league_json / all_entries exist.")

df = pd.DataFrame(rows) #convert the JSON data into a dataframe 
n_src = len(rows) #number of records in the JSON data
n_df = len(df) #number of records in the dataframe
cols = len(df.columns) #number of columns in the dataframe
if n_src == 0:
    print("Conversion finished: empty DataFrame (0 rows) — no API data in rows yet.")
elif n_df == n_src:
    print(
        f"JSON → DataFrame conversion successful: {n_df} rows × {cols} columns "
        f"(matches all {n_src} records from summoner_league_json / all_entries)."
    )
else:
    print(
        f"Unexpected row count after conversion: DataFrame has {n_df} rows but source list has {n_src}."
    )


JSON → DataFrame conversion successful: 52729 rows × 12 columns (matches all 52729 records from summoner_league_json / all_entries).


---
### 3. Analyzing the DataFrame with Pandas Operations
#### 3a. What kinds of data did we actually fetch with all these API requests? Finding out with `df.head()`
This operation will allow me to see all the different kinds of data associated with each row that was fetched from the API. Here, we can see that in addition to the PUUID (which we'll need to fetch match history data), we can also see their associated LP (which determines rank), wins, losses, veteran status, activity, and win streaks. 

In [ ]:
# `df` is built in section 2 — run that cell first.
print("shape (rows, columns):", df.shape) #display the number of rows and columns in the dataframe
df.head() #display the first 5 rows of the dataframe

shape (rows, columns): (52729, 12)


,leagueId,queueType,tier,rank,puuid,leaguePoints,wins,losses,veteran,inactive,freshBlood,hotStreak
0,04116fb7-6868-4521-9910-61fb90b76e5d,RANKED_SOLO_5x5,BRONZE,IV,dmr_PvsJDqRYwdOsAvRCGlp5676GgstHkcKjev1N03OTVQ...,18,33,52,False,False,False,False
1,1ec3dd38-bf92-4c0e-aa5d-ca540bd70919,RANKED_SOLO_5x5,BRONZE,IV,Q5tPr4VoHQO40KOqRSjQMkb10Uw1uFX62YNk0ZJNYLR4F3...,23,5,8,False,False,False,False
2,3610058c-0fc6-4ee5-a161-dad705389f88,RANKED_SOLO_5x5,BRONZE,IV,hseHG45GLA7RwEdsMcqt1t6VjpEMPlV_Jelsv26jLRWCQy...,31,36,63,False,False,False,False
3,346389f6-9c92-414f-b2a1-19f5524e4895,RANKED_SOLO_5x5,BRONZE,IV,LBIg2k5fCNbUGMESy8Gh5hJH9ZupZGwKdwodsu7rNwdKha...,15,169,202,False,False,False,False
4,04ec97a8-c7cc-4d22-aa8d-24ac7b697054,RANKED_SOLO_5x5,BRONZE,IV,W8zW_QFs2RAV1PcYUYWP31upsQoaVDpOmVAuUDeZyWLsV1...,18,31,40,False,False,False,False


#### 3b. How many times does each PUUID appear? Using the operations: `df['puuid'].value_counts()` & `df.duplicated(subset=["puuid"]).sum()` 
This operation will allow me to see how many times a PUUID appears in the dataframe. Because I had to fetch data from the API in two different sesions (once before receiving an HTTP 429 error, and once after fixing it), there's a possibility that I accidentally fetched the same pages twice. Since PUUIDs are supposed to be unique to each account, each PUUID should really only appear once in my dataframe. This operation will allow me to catch if I accidentally fetched the same pages twice. 

In [ ]:
df['puuid'].value_counts() #display the number of times each PUUID appears in the dataframe in descending order

puuid
ii89RSci7pZ3Jv6TdPagBbbQVEZzFd2llBH9BLhIipO18RCpJTnt0aFxrXwBOKIN7NouuTy07GtqiQ    2
B1TB7XUm8qYdS5zFTmtOAXlQTAhahPlb0Uhb-1ZPS741W-foGbeiZpp242p0YpZvOcGniZZhBhz--A    2
7IOzK6xmUEZ1-zegE26isH9O5FsFKKigiP9rzhd0eoG6N-y5wMob-vOhFBUG0HuDmnn-8vtOzop8AQ    2
ableOWCSRBhCLXyxj-ntkSCrsRxCx4XDd8xhEUoCtsV48_4PI2iAAfbWe6bdlHNsjf1D6nXSNUQ2xg    2
o-b2eX8DQHUqnt1JtChX99atA5kVT6fdA7STnjwqugvtxTl30cRnEEHQlazAAD8FcqXXiKh9MQFfuQ    2
                                                                                 ..
vd37E5n0iok6pQmUbdmtDAH9CLr02BGqjZTQQSZOXOd-N8mZflR76ccZZs_tLlOQ2Kk_XpUF4h86RA    1
DN7Y4yEoj0S3UglHmN1sBe8Is_Ru6aPsjd1KO0Oyej8FLl4tpOCulK8e5F_cF60nr8pyY5lx1w8KkA    1
7iOcpIxTZaQu8ZezW3ciIr59C8dpM_pTudfh9Kalx2yyZSgzPZmHzlrtAj1liYcqpFI2ngWXiyf5Mw    1
WFl6wxhqyNJCGx_6m3QWV9OeaJSqBqVhESi4QJhYb3p-P5ByMlBrz-Y20mtsPjEvpSqs91u-Berfbg    1
0DzEUblGPo-tTMNDC17gD3Qn_YcbOgtVJ_mWDSpzmjSzYqdS5h1RJqoBmfcOf1PwLxhfw8_EcFCsWw    1
Name: count, Length: 51447, dtype: int64

This next code block will tell us how many times a duplicated PUUID appears in the dataframe. 

In [12]:
df.duplicated(subset=["puuid"]).sum() #display the number of duplicate PUUIDs in the dataframe

np.int64(1282)

Here, we can see that there are 1282 duplicate entries in the dataframe. 

#### 3c. How can we merge or deduplicate the repeated PUUIDs? Cleaning with `df.drop_duplicates` and checking with `df_deduped.duplicated`
Now, our next step is to merge or deduplicate this repeated data.

In [ ]:
df_deduped = df.drop_duplicates(subset=["puuid"], keep="last") #keeps the last occurence of each PUUID, which should've come from the second API call where all pages were fetched successfully

#check if the deduplication worked
print("rows before:", len(df))
df_deduped = df.drop_duplicates(subset=["puuid"], keep="first")
print("rows after:", len(df_deduped))

df_deduped.duplicated(subset=["puuid"]).sum() #double check that the deduplication worked by checking the number of duplicate PUUIDs in the dataframe
#make sure you're using the deduped dataframe instead of the original one! original one still has the duplicates 

rows before: 52729
rows after: 51447


np.int64(0)

Now that our dataset has been cleaned of any duplicates, we can use other Pandas Analysis operations to learn more about the dataframe.

#### 3d. Does this dataframe have any NaN values that could affect our future data analysis? Finding out with `df_deduped.isnull().sum()`

Because we know that PUUIDs are unique to each account, each column should be filled out with information that comes from Riot's API. By using this operation, we can double check that there won't be any NaN values that could affect our data analysis. 

In [22]:
### 4. Extracting PUUIDs to use in another API for Match IDs
df_deduped.isnull().sum()

leagueId        0
queueType       0
tier            0
rank            0
puuid           0
leaguePoints    0
wins            0
losses          0
veteran         0
inactive        0
freshBlood      0
hotStreak       0
dtype: int64

Because we really only need the PUUIDs from the dataframe to proceed, we can stop our Pandas Analysis here. 

Now that we've cleaned up the duplicate data and gained a good understanding of the dataframe, we can move one to extract PUUIDs from each row and use them to fetch data about Match Id, which will give us the suitable information to fetch data about Champion Picks. 

---
### 5. Extracting PUUIDs to use in another API for Match IDs 

In [21]:
# Extract unique PUUIDs from the deduplicated DataFrame for downstream API scripts.
if "df_deduped" not in globals():
    raise RuntimeError("Run the deduplication cell first so df_deduped exists.")

puuid_list = df_deduped["puuid"].dropna().astype(str).tolist()
print(f"Prepared {len(puuid_list)} PUUIDs for API scripting.")
print("Sample:", puuid_list[:5])

# Optional: save to disk as JSON so another script can load it directly.
with open("puuid_list.json", "w", encoding="utf-8") as f:
    json.dump(puuid_list, f, ensure_ascii=False, indent=2)
print("Saved puuid_list.json")

Prepared 51447 PUUIDs for API scripting.
Sample: ['dmr_PvsJDqRYwdOsAvRCGlp5676GgstHkcKjev1N03OTVQyvTqLUTZmpYmQnR1QetaEeFBBAFDiI0A', 'Q5tPr4VoHQO40KOqRSjQMkb10Uw1uFX62YNk0ZJNYLR4F3k5hRus6RZxNDNbblocQhEc4or40vRRtA', 'hseHG45GLA7RwEdsMcqt1t6VjpEMPlV_Jelsv26jLRWCQyA8wqK22JG-C9hUTz_A5ilXbKCjvt-dMw', 'LBIg2k5fCNbUGMESy8Gh5hJH9ZupZGwKdwodsu7rNwdKhaOy5_ZjEDJ9MWwKmWcf35kMgsWxGpRqrg', 'W8zW_QFs2RAV1PcYUYWP31upsQoaVDpOmVAuUDeZyWLsV1_sRvTcjUaq-xjZKKhhUrwfkLWgcbmSIQ']
Saved puuid_list.json


---
### 6. Using stored PUUIDs to fetch data for Match IDs 
It's important to be precise with this API, because it could return matches that weren't played in Ranked 5x5 Solo (which is how we filtered the PUUIDs above). For analysis purposes, we'll be fetching the 10 most recent matches associated with the PUUIDs that were extracted from the dataframe above. As a reminder, the PUUIDs are supposed to represent players that were Bronze IV in Ranked 5x5 Solo queue as of May 6th, 2026. 

Additionally, even though the JSON contains around 51,400 PUUIDs (as fetched from the previous API endpoint), the code below will only fetch Match IDs for the first 500 PUUIDs in the JSON file due to time constraints and request limits from Riot. A sample size of 500 should be statistically significant enough to demonstrate major patterns in player behavior. This will take around 40 minutes.

In [ ]:
import json
import os
import time
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path

QUEUE_ID = 420  # Ranked Solo/Duo 5v5 — see https://static.developer.riotgames.com/docs/lol/queues.json
MATCH_IDS_START = 0
MATCH_IDS_COUNT = 10
REQUEST_SLEEP_SEC = 2.0  # increase if you get HTTP 429
MAX_PUUIDS = 500  # loop through the first 500 PUUIDs from puuid_list.json

#function to load .env file so that this code can read the API key on it 
def _load_env(path: Path) -> None:
    if not path.is_file():
        return
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, val = line.partition("=")
        key, val = key.strip(), val.strip().strip('"').strip("'")
        if key:
            os.environ.setdefault(key, val)

# Resolve Week 5/.env whether the kernel cwd is the repo root or the Week 5 folder
for _env in (Path(".env"), Path("Week 5") / ".env"):
    _load_env(_env)

api_key = os.environ.get("RIOT_API_KEY", "").strip() #the api key I stored in .env, will be regenerated every 24 hours
if not api_key:
    raise ValueError("RIOT_API_KEY missing: add it to Week 5/.env or set it in the environment.") #debugging flag in case API key is missing

# Load PUUID list using relative fallback paths so this works from different kernel cwd values  bc having trouble that this code is not reading the puuid_list.json file
puuid_candidates = [
    Path("puuid_list.json"),
    Path("Week 5") / "puuid_list.json",
    Path("..") / "Week 5" / "puuid_list.json",
]
puuid_path = next((p for p in puuid_candidates if p.is_file()), None)
if puuid_path is None: #debugging flag in case puuid_list.json is not found
    raise FileNotFoundError(
        "puuid_list.json not found. Tried: " + ", ".join(str(p) for p in puuid_candidates)
    )
#load the puuid_list.json file
with open(puuid_path, encoding="utf-8") as f:
    puuids = json.load(f)

if not isinstance(puuids, list): #debugging flag in case puuid_list.json is not a list
    raise TypeError("puuid_list.json should contain a JSON array of PUUID strings.")

if MAX_PUUIDS is not None: 
    puuids = puuids[: int(MAX_PUUIDS)]

# Save checkpoint/output in a relative path that is valid for current cwd bc having trouble that this code is not reading the puuid_list.json file
out_candidates = [
    Path("match_ids_by_puuid.json"),
    Path("Week 5") / "match_ids_by_puuid.json",
    Path("..") / "Week 5" / "match_ids_by_puuid.json",
]
out_path = next((p for p in out_candidates if p.parent.exists()), out_candidates[0])

# Resume support: load existing checkpoint and skip those PUUIDs on rerun in case session is interrupted.
if out_path.is_file():
    with open(out_path, encoding="utf-8") as f:
        existing = json.load(f)
    if not isinstance(existing, dict):
        raise TypeError("match_ids_by_puuid.json should contain a JSON object keyed by puuid.")
    match_ids_by_puuid: dict[str, list] = existing
else:
    match_ids_by_puuid: dict[str, list] = {}

#create a set of PUUIDs to compare with the existing PUUIDs in the JSON file
puuid_set = set(puuids)
completed = {k for k in match_ids_by_puuid.keys() if k in puuid_set}
remaining_puuids = [p for p in puuids if p not in completed]

#print the number of PUUIDs that have already been completed and the number of PUUIDs that are remaining to be completed
total_puuids = len(puuids)
remaining_total = len(remaining_puuids)
print(f"Already completed: {len(completed)} / {total_puuids}")
print(f"Remaining this run: {remaining_total}")

#set the headers for the API request
headers = {
    "X-Riot-Token": api_key,
    "User-Agent": "HCDE530-A5/1.0 (match-v5 by-puuid; coursework)",
}
base_url = "https://americas.api.riotgames.com/lol/match/v5/matches/by-puuid" #endpoint for fetching match IDs by PUUID

#loop through the remaining PUUIDs and fetch the match IDs
for step_idx, puuid in enumerate(remaining_puuids, start=1):
    safe_puuid = urllib.parse.quote(puuid, safe="")
    query = urllib.parse.urlencode(
        {"start": MATCH_IDS_START, "count": MATCH_IDS_COUNT, "queue": QUEUE_ID} #parameters for the API request
    )
    url = f"{base_url}/{safe_puuid}/ids?{query}" #construct the URL for the API request
    req = urllib.request.Request(url, headers=headers) #send the request to the API

    try:
        with urllib.request.urlopen(req, timeout=30) as resp: 
            match_ids = json.load(resp)
    except urllib.error.HTTPError as e:
        body = e.read().decode(errors="replace")
        if e.code == 429:
            time.sleep(60)
            with urllib.request.urlopen(req, timeout=30) as resp:
                match_ids = json.load(resp)
        else:
            raise RuntimeError(f"HTTP {e.code} {e.reason}: {body[:500]}") from e

    match_ids_by_puuid[puuid] = match_ids #add the match IDs to the dictionary

    # Checkpoint after each success so reruns can resume where this run left off.
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(match_ids_by_puuid, f, ensure_ascii=False, indent=2)

    processed_now = step_idx
    processed_total = len(completed) + processed_now
    print(
        f"PUUIDs processed this run: {processed_now} / {remaining_total} | "
        f"overall: {processed_total} / {total_puuids}",
        flush=True,
    )
    time.sleep(REQUEST_SLEEP_SEC) #wait 2 seconds between requests to avoid overwhelming the server

print(f"Done. Saved {len(match_ids_by_puuid)} PUUID entries to {out_path.resolve()}") #print the number of PUUIDs that have been processed and the total number of PUUIDs that have been processed

Already completed: 480 / 500
Remaining this run: 20
PUUIDs processed this run: 1 / 20 | overall: 481 / 500
PUUIDs processed this run: 2 / 20 | overall: 482 / 500
PUUIDs processed this run: 3 / 20 | overall: 483 / 500
PUUIDs processed this run: 4 / 20 | overall: 484 / 500
PUUIDs processed this run: 5 / 20 | overall: 485 / 500
PUUIDs processed this run: 6 / 20 | overall: 486 / 500
PUUIDs processed this run: 7 / 20 | overall: 487 / 500
PUUIDs processed this run: 8 / 20 | overall: 488 / 500
PUUIDs processed this run: 9 / 20 | overall: 489 / 500
PUUIDs processed this run: 10 / 20 | overall: 490 / 500
PUUIDs processed this run: 11 / 20 | overall: 491 / 500
PUUIDs processed this run: 12 / 20 | overall: 492 / 500
PUUIDs processed this run: 13 / 20 | overall: 493 / 500
PUUIDs processed this run: 14 / 20 | overall: 494 / 500
PUUIDs processed this run: 15 / 20 | overall: 495 / 500
PUUIDs processed this run: 16 / 20 | overall: 496 / 500
PUUIDs processed this run: 17 / 20 | overall: 497 / 500
PUUID

#### 6a. Converting Match ID JSON to usable dataframe
Similar to the initial dataframe that allowed us to extract the PUUIDs, now we need to convert the Match IDs JSON into a usable dataframe for Pandas Analysis.

In [ ]:
from pathlib import Path
import json

# Find the match-id checkpoint file using relative fallbacks.
match_candidates = [
    Path("match_ids_by_puuid.json"),
    Path("Week 5") / "match_ids_by_puuid.json",
    Path("..") / "Week 5" / "match_ids_by_puuid.json",
]
match_path = next((p for p in match_candidates if p.is_file()), None)
if match_path is None:
    raise FileNotFoundError( #debugging flag in case match_ids_by_puuid.json is not found
        "match_ids_by_puuid.json not found. Tried: " + ", ".join(str(p) for p in match_candidates)
    )

#load the match_ids_by_puuid.json file
with open(match_path, encoding="utf-8") as f:
    match_ids_by_puuid = json.load(f)

if not isinstance(match_ids_by_puuid, dict):
    raise TypeError("match_ids_by_puuid.json should contain an object mapping puuid -> list[match_id].")

# Flatten dict-of-lists into row records for pandas.
rows = []
for puuid, match_ids in match_ids_by_puuid.items():
    if not isinstance(match_ids, list):
        continue
    for match_id in match_ids:
        rows.append({"puuid": puuid, "match_id": match_id})

matches_df = pd.DataFrame(rows) #convert the match_ids_by_puuid.json file into a dataframe

print("Loaded:", match_path.resolve())
print("shape (rows, columns):", matches_df.shape)
print("unique PUUIDs:", matches_df["puuid"].nunique() if not matches_df.empty else 0)
print("unique match IDs:", matches_df["match_id"].nunique() if not matches_df.empty else 0)
matches_df.head()

Loaded: /Users/ruofuli/hcde530/Week 5/match_ids_by_puuid.json
shape (rows, columns): (4831, 2)
unique PUUIDs: 500
unique match IDs: 4825


,puuid,match_id
0,dmr_PvsJDqRYwdOsAvRCGlp5676GgstHkcKjev1N03OTVQ...,NA1_5550304472
1,dmr_PvsJDqRYwdOsAvRCGlp5676GgstHkcKjev1N03OTVQ...,NA1_5545162641
2,dmr_PvsJDqRYwdOsAvRCGlp5676GgstHkcKjev1N03OTVQ...,NA1_5530609519
3,dmr_PvsJDqRYwdOsAvRCGlp5676GgstHkcKjev1N03OTVQ...,NA1_5530552256
4,dmr_PvsJDqRYwdOsAvRCGlp5676GgstHkcKjev1N03OTVQ...,NA1_5530519318


---
### 7. Analyzing the Match ID dataframe (named `matches_df) using Pandas Analysis
#### 7a. How many repeated Match IDs are there? With `matches_df.duplicated` and `matches_df["match_id"].value_counts()`
It makes sense if there are repeated Match IDs, because it's entirely possible that some of these players played against each other in multiple matches. However, because we want to eventually find out which Support role champion has the highest pick rate, it's best to (once again) deduplicate our repeated Match IDs to avoid skewing our data. 

In [29]:
if "matches_df" not in globals():
    raise RuntimeError("Run section 6a first so matches_df exists.")

# Count duplicate rows by match_id (all repeats beyond the first occurrence).
duplicate_match_rows = matches_df.duplicated(subset=["match_id"]).sum()

# Count how many distinct match IDs appear more than once.
match_id_counts = matches_df["match_id"].value_counts()
repeated_match_id_count = (match_id_counts > 1).sum()

print("Total rows:", len(matches_df))
print("Unique match IDs:", matches_df["match_id"].nunique())
print("Duplicate match-ID rows:", duplicate_match_rows)
print("Distinct match IDs that are repeated:", repeated_match_id_count)

# Optional: preview top repeated match IDs.
match_id_counts[match_id_counts > 1].head(10)

Total rows: 4831
Unique match IDs: 4825
Duplicate match-ID rows: 6
Distinct match IDs that are repeated: 6


match_id
NA1_5553492411    2
NA1_5553880975    2
NA1_5477932306    2
NA1_5515936514    2
NA1_5554756967    2
NA1_5554481168    2
Name: count, dtype: int64

Honestly, super surprised that only 6 Match IDs were repeated. This means that across the 10 most recent games (as of the snapshot taken on May 6, 2026) from the 500 player sample size I used, only 12 of these entries appeared in the same match with 1 other entry in this sample. 

Now I'm curious... 

#### 7b. Which of these PUUIDs shared the same Match ID? With `matches_df.groupby("match_id")["puuid"]` 

In [ ]:
if "matches_df" not in globals():
    raise RuntimeError("Run section 6a first so matches_df exists.")

# Show match IDs that were linked to more than one PUUID, plus the PUUIDs in each shared match.
shared_match_puuids = (
    matches_df.groupby("match_id")["puuid"]
    .agg(lambda s: sorted(set(s)))
    .reset_index(name="puuids")
)
shared_match_puuids["puuid_count"] = shared_match_puuids["puuids"].str.len() #count the number of PUUIDs in each match
shared_match_puuids = shared_match_puuids[shared_match_puuids["puuid_count"] > 1] #only keep the matches that have more than 1 PUUID
shared_match_puuids = shared_match_puuids.sort_values("puuid_count", ascending=False) #sort the matches by the number of PUUIDs in descending order

print("match IDs shared by multiple PUUIDs:", len(shared_match_puuids))
shared_match_puuids.head(20)

# One row per (match_id, puuid) pair for easier filtering/export.
shared_match_pairs = shared_match_puuids[["match_id", "puuids"]].explode("puuids")#explode the PUUIDs into separate rows
shared_match_pairs = shared_match_pairs.rename(columns={"puuids": "puuid"}) #rename the PUUID column to "puuid"
shared_match_pairs.head(20) #first 20 entries but there's only 12 rows in total

match IDs shared by multiple PUUIDs: 6


,match_id,puuid
1169,NA1_5477932306,kM2Q35kVATyg_-g2p4hucag6ZSM57O8VIIs6rJKO4rH3sw...
1169,NA1_5477932306,romsuHz5QKq_XUkx7N4qKPjoc2vD4VHsbgFlGeBlmYyufv...
2463,NA1_5515936514,N5ad-8s5iKPYbF14er-6c8TcorXz5JpAkWZ4jZjbcSeR-Y...
2463,NA1_5515936514,gbLR3Z4L37vx7U3zu5hYHsFHCjFg9seX5KGXOTTchCkdYJ...
4329,NA1_5553492411,FFuw8Efm1P2O9fvvt_xSrn9kOfHI5EaaGnFTBJ3sTN0PnD...
4329,NA1_5553492411,eoraV-WPRU9WFjZV_MR7t-6ArYKxPMi3wwRHn7tByfnNiT...
4418,NA1_5553880975,HBEhi9sBqE7VqfS1OmK6UcJ-0K6ob-_1LYQ6Q-jVzrH2e9...
4418,NA1_5553880975,lAMJFfS183Y5omawvwncjvaJiOBSOMeDaJZD2dv-8IdtQ2...
4585,NA1_5554481168,44QNoWysHHA5AjcZ3gmo05AdxqNQGYeV0Bz5vqrUcxqewd...
4585,NA1_5554481168,EmW71KcXaKFFJih3Y2F2Hbuq1ls3-4PFU3-F5jixCeIdlO...


#### 7c. Deduplicating repeated Match IDs
Back to business. As mentioned in 7a, every Match ID has to be unique to avoid skewing the data when we have to analyze it later. 

In [ ]:
if "matches_df" not in globals():
    raise RuntimeError("Run section 6a first so matches_df exists.")

rows_before = len(matches_df)

# Deduplicate by match_id so each match contributes only once to later analysis.
matches_df_deduped = matches_df.drop_duplicates(subset=["match_id"], keep="first").copy()

rows_after = len(matches_df_deduped)#number of rows in the deduplicated dataframe
removed = rows_before - rows_after #number of rows removed from the deduplicated dataframe

print("rows before dedupe:", rows_before)
print("rows after dedupe:", rows_after)
print("duplicate rows removed:", removed)
print("remaining duplicate match IDs:", matches_df_deduped.duplicated(subset=["match_id"]).sum())

# Save deduped match IDs for downstream scripts/notebooks.
match_deduped_candidates = [
    Path("matches_df_deduped.json"), #possible file paths to find the deduplicated dataframe for match IDs
    Path("Week 5") / "matches_df_deduped.json",
    Path("..") / "Week 5" / "matches_df_deduped.json",
]
match_deduped_out = next((p for p in match_deduped_candidates if p.parent.exists()), match_deduped_candidates[0]) #find the first file path that exists
matches_df_deduped.to_json(match_deduped_out, orient="records", indent=2) #save the deduplicated dataframe to the file path
print("Saved deduped match DataFrame to:", match_deduped_out.resolve()) #print the file path of the deduplicated dataframe

matches_df_deduped.head()

rows before dedupe: 4831
rows after dedupe: 4825
duplicate rows removed: 6
remaining duplicate match IDs: 0
Saved deduped match DataFrame to: /Users/ruofuli/hcde530/Week 5/matches_df_deduped.json


,puuid,match_id
0,dmr_PvsJDqRYwdOsAvRCGlp5676GgstHkcKjev1N03OTVQ...,NA1_5550304472
1,dmr_PvsJDqRYwdOsAvRCGlp5676GgstHkcKjev1N03OTVQ...,NA1_5545162641
2,dmr_PvsJDqRYwdOsAvRCGlp5676GgstHkcKjev1N03OTVQ...,NA1_5530609519
3,dmr_PvsJDqRYwdOsAvRCGlp5676GgstHkcKjev1N03OTVQ...,NA1_5530552256
4,dmr_PvsJDqRYwdOsAvRCGlp5676GgstHkcKjev1N03OTVQ...,NA1_5530519318


While I was expecting 5000 initial rows, there ended up being less because some players didn't play enough Ranked games to fill up all 10 spots. 

---
### 8. Using stored Match IDs to fetch data for Champion Picks
In our third and final API that we'll be calling to answer our analytical questions, we'll be using the Match IDs stored in the deduped JSON file in another endpoint that fetches data for which champions were selected per Match ID.

Similar to the transition from PUUIDs to Match IDs, I will be unable to cycle through all of the Match IDs that were fetched previously due to API request limits. As a result, the code below will only cycle through the first 500 Match IDs that appear in the JSON file, which equals to about 50 players.

Each Match ID is going to return a lot of fields, so we'll have to do some further data cleaning after to extract the fields that we actually care about; namely, the 10 champion picks for the game, 5 from each team. The game doesn't allow for repeated champion picks (at least in Ranked), so each champion should be unique. 

In [1]:
import json
import os
import time
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path

MAX_MATCH_IDS = 500  # first N unique match IDs from matches_df_deduped.json
REQUEST_SLEEP_SEC = 2.0  # increase if you get HTTP 429


def _load_env(path: Path) -> None:
    if not path.is_file():
        return
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, val = line.partition("=")
        key, val = key.strip(), val.strip().strip('"').strip("'")
        if key:
            os.environ.setdefault(key, val)


def _extract_champion_picks(match_data: dict) -> list[dict]:
    """Pull champion/role fields from info.participants for one match."""
    participants = match_data.get("info", {}).get("participants", [])
    picks = []
    for p in participants:
        if not isinstance(p, dict):
            continue
        picks.append(
            {
                "puuid": p.get("puuid"),
                "championId": p.get("championId"),
                "championName": p.get("championName"),
                "teamId": p.get("teamId"),
                "teamPosition": p.get("teamPosition"),
                "individualPosition": p.get("individualPosition"),
                "win": p.get("win"),
            }
        )
    return picks


for _env in (Path(".env"), Path("Week 5") / ".env"):
    _load_env(_env)

api_key = os.environ.get("RIOT_API_KEY", "").strip()
if not api_key:
    raise ValueError("RIOT_API_KEY missing: add it to Week 5/.env or set it in the environment.")

deduped_candidates = [
    Path("matches_df_deduped.json"),
    Path("Week 5") / "matches_df_deduped.json",
    Path("..") / "Week 5" / "matches_df_deduped.json",
]
deduped_path = next((p for p in deduped_candidates if p.is_file()), None)
if deduped_path is None:
    raise FileNotFoundError(
        "matches_df_deduped.json not found. Tried: " + ", ".join(str(p) for p in deduped_candidates)
    )

with open(deduped_path, encoding="utf-8") as f:
    deduped_records = json.load(f)

if not isinstance(deduped_records, list):
    raise TypeError("matches_df_deduped.json should contain a JSON array of records.")

seen: set[str] = set()
match_ids: list[str] = []
for row in deduped_records:
    mid = row.get("match_id") if isinstance(row, dict) else None
    if not mid or mid in seen:
        continue
    seen.add(mid)
    match_ids.append(str(mid))
    if len(match_ids) >= MAX_MATCH_IDS:
        break

out_candidates = [
    Path("champion_picks_by_match_id.json"),
    Path("Week 5") / "champion_picks_by_match_id.json",
    Path("..") / "Week 5" / "champion_picks_by_match_id.json",
]
out_path = next((p for p in out_candidates if p.parent.exists()), out_candidates[0])

if out_path.is_file():
    with open(out_path, encoding="utf-8") as f:
        existing = json.load(f)
    if not isinstance(existing, dict):
        raise TypeError("champion_picks_by_match_id.json should be a JSON object keyed by match_id.")
    champion_picks_by_match_id: dict[str, list] = existing
else:
    champion_picks_by_match_id = {}

completed = {mid for mid in match_ids if mid in champion_picks_by_match_id}
remaining_match_ids = [mid for mid in match_ids if mid not in completed]

total_match_ids = len(match_ids)
remaining_total = len(remaining_match_ids)
print("Loaded match list from:", deduped_path.resolve())
print("Target unique match IDs:", total_match_ids)
print("Already completed:", len(completed), "/", total_match_ids)
print("Remaining this run:", remaining_total)
print("Checkpoint file:", out_path.resolve())

headers = {
    "X-Riot-Token": api_key,
    "User-Agent": "HCDE530-A5/1.0 (match-v5 champion picks; coursework)",
}
base_url = "https://americas.api.riotgames.com/lol/match/v5/matches"

for step_idx, match_id in enumerate(remaining_match_ids, start=1):
    safe_match_id = urllib.parse.quote(match_id, safe="")
    url = f"{base_url}/{safe_match_id}"
    req = urllib.request.Request(url, headers=headers)

    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            match_data = json.load(resp)
    except urllib.error.HTTPError as e:
        body = e.read().decode(errors="replace")
        if e.code == 429:
            time.sleep(60)
            with urllib.request.urlopen(req, timeout=30) as resp:
                match_data = json.load(resp)
        else:
            raise RuntimeError(f"HTTP {e.code} {e.reason}: {body[:500]}") from e

    champion_picks_by_match_id[match_id] = _extract_champion_picks(match_data)

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(champion_picks_by_match_id, f, ensure_ascii=False, indent=2)

    processed_now = step_idx
    processed_total = len(completed) + processed_now
    print(
        f"Match IDs processed this run: {processed_now} / {remaining_total} | "
        f"overall: {processed_total} / {total_match_ids}",
        flush=True,
    )
    time.sleep(REQUEST_SLEEP_SEC)

print(f"Done. Saved champion picks for {len(champion_picks_by_match_id)} matches to {out_path.resolve()}")

Loaded match list from: /Users/ruofuli/hcde530/Week 5/matches_df_deduped.json
Target unique match IDs: 500
Already completed: 0 / 500
Remaining this run: 500
Checkpoint file: /Users/ruofuli/hcde530/Week 5/champion_picks_by_match_id.json
Match IDs processed this run: 1 / 500 | overall: 1 / 500
Match IDs processed this run: 2 / 500 | overall: 2 / 500
Match IDs processed this run: 3 / 500 | overall: 3 / 500
Match IDs processed this run: 4 / 500 | overall: 4 / 500
Match IDs processed this run: 5 / 500 | overall: 5 / 500
Match IDs processed this run: 6 / 500 | overall: 6 / 500
Match IDs processed this run: 7 / 500 | overall: 7 / 500
Match IDs processed this run: 8 / 500 | overall: 8 / 500
Match IDs processed this run: 9 / 500 | overall: 9 / 500
Match IDs processed this run: 10 / 500 | overall: 10 / 500
Match IDs processed this run: 11 / 500 | overall: 11 / 500
Match IDs processed this run: 12 / 500 | overall: 12 / 500
Match IDs processed this run: 13 / 500 | overall: 13 / 500
Match IDs proc

The code above is actually the second iteration of the script to fetch data from the endpoint due to its massive payload. The first time, I had asked the script to store ALL the fetched data into a JSON file, but that quickly crashed Cursor due to how much information there was. The second iteration of this code still fetches all the API data associated with this Match ID from this endpoint, but now only stores the champion pick data after it's already been sorted for it. 

#### 8a. Converting champion pick JSON to a usable DataFrame
Next, it'd be really nice if we could see our data laid out. The script below converts `champion_picks_by_match_id` into a usable DataFrame for Pandas Analysis. 

In [6]:
from pathlib import Path
import json

picks_candidates = [
    Path("champion_picks_by_match_id.json"),
    Path("Week 5") / "champion_picks_by_match_id.json",
    Path("..") / "Week 5" / "champion_picks_by_match_id.json",
]
picks_path = next((p for p in picks_candidates if p.is_file()), None)
if picks_path is None:
    raise FileNotFoundError(
        "champion_picks_by_match_id.json not found. Tried: " + ", ".join(str(p) for p in picks_candidates)
    )

with open(picks_path, encoding="utf-8") as f:
    champion_picks_by_match_id = json.load(f)

if not isinstance(champion_picks_by_match_id, dict):
    raise TypeError("champion_picks_by_match_id.json should be a JSON object keyed by match_id.")

rows = []
for match_id, participants in champion_picks_by_match_id.items():
    if not isinstance(participants, list):
        continue
    for pick in participants:
        if not isinstance(pick, dict):
            continue
        row = {"match_id": match_id, **pick}
        rows.append(row)

champion_picks_df = pd.DataFrame(rows)

print("Loaded:", picks_path.resolve())
print("shape (rows, columns):", champion_picks_df.shape)
champion_picks_df.head(10)

Loaded: /Users/ruofuli/hcde530/Week 5/champion_picks_by_match_id.json
shape (rows, columns): (4990, 8)


,match_id,puuid,championId,championName,teamId,teamPosition,individualPosition,win
0,NA1_5550304472,dmr_PvsJDqRYwdOsAvRCGlp5676GgstHkcKjev1N03OTVQ...,6,Urgot,100,TOP,TOP,False
1,NA1_5550304472,73-lrrlMMVWme9MM0yPNZGWRe74f6bjBBK5HAiZu3W9Ddf...,11,MasterYi,100,JUNGLE,JUNGLE,False
2,NA1_5550304472,jK6YPX1S57Juet21uvU4wLeRrINqVQ-7PWF7Qk_zEQgugc...,79,Gragas,100,MIDDLE,MIDDLE,False
3,NA1_5550304472,iVw8Qyyy0kgySNltkzIj9Cer2Z-vOFtc1pHUwtNjNI9tsV...,875,Sett,100,BOTTOM,BOTTOM,False
4,NA1_5550304472,UL3qXsy63hmThQw39LTtPWK2JI316v5jHX8r_yw5g9A873...,235,Senna,100,UTILITY,UTILITY,False
5,NA1_5550304472,Au4sDCmV_atawjHQC2vcrWMPjWuc8PJOESsfdSsjd8ccZp...,23,Tryndamere,200,TOP,TOP,True
6,NA1_5550304472,0rYwoUgz9vPFfOBod9IatryWC116iveVrTyuNmpSbhBaY8...,141,Kayn,200,JUNGLE,JUNGLE,True
7,NA1_5550304472,8s1zQmQWiAEgiwPnj1SOukd-e31Em6ZJrXFMcaqKcp1sAu...,134,Syndra,200,MIDDLE,MIDDLE,True
8,NA1_5550304472,pHmzT3fviEQn8RsZTB4P3V2p2K-5SjbMZEFFt7S5i1LwXD...,145,Kaisa,200,BOTTOM,BOTTOM,True
9,NA1_5550304472,o9UXKEi4u2kwH6F9UJVxFy8x6Jbi2tXl3Y5sFmPaDBR8Us...,89,Leona,200,UTILITY,UTILITY,True


#### 8b. Further Cleaning- Sorting by teamPosition == UTILITY
Even though we already sorted the fetched data and only stored the information about champion picks, it still includes a lot of other roles that we're not looking for. As a result, let's clean this dataset up a little to only show us the Support champion picks. 

In [9]:
if "champion_picks_df" not in globals():
    raise RuntimeError("Run section 8a first so champion_picks_df exists.")

support_picks_df = champion_picks_df[champion_picks_df["teamPosition"] == "UTILITY"].copy()

print("rows before filter:", len(champion_picks_df))
print("support rows (UTILITY):", len(support_picks_df))
support_picks_df.head(10)

rows before filter: 4990
support rows (UTILITY): 996


,match_id,puuid,championId,championName,teamId,teamPosition,individualPosition,win
4,NA1_5550304472,UL3qXsy63hmThQw39LTtPWK2JI316v5jHX8r_yw5g9A873...,235,Senna,100,UTILITY,UTILITY,False
9,NA1_5550304472,o9UXKEi4u2kwH6F9UJVxFy8x6Jbi2tXl3Y5sFmPaDBR8Us...,89,Leona,200,UTILITY,UTILITY,True
14,NA1_5545162641,iAPisvb-m-cDkpVmzVJtNKdNjTwoIgk-PkKIvRQScvPeed...,53,Blitzcrank,100,UTILITY,UTILITY,True
19,NA1_5545162641,z47LTXaQxM72C5dDdLwpdhuZJVDjcmnqkjE11WKKhqtDPx...,26,Zilean,200,UTILITY,UTILITY,False
24,NA1_5530609519,Kj3raKvthe46haxKC6N2LOpUK8_VxgOyRUo48bufObv60i...,22,Ashe,100,UTILITY,UTILITY,True
29,NA1_5530609519,LkD9mUW8W5xEDss269mnh1dmIhdsc-kwm3JB7B_YU8bunW...,50,Swain,200,UTILITY,UTILITY,False
34,NA1_5530552256,YGIe4YJJm9C8UxGgFnMxB0R28D8eoq7sZd4_7LScRtXGW0...,223,TahmKench,100,UTILITY,UTILITY,False
39,NA1_5530552256,vlZi8mpfS47IqAtxbXCjLUWUq9HZqzumvz1fZQ-sYeYxUn...,50,Swain,200,UTILITY,UTILITY,True
44,NA1_5530519318,05ULVbcqTZ6gRRf_HohaNzkVKlnpa-_X1vScIiIOc2pReR...,161,Velkoz,100,UTILITY,UTILITY,True
49,NA1_5530519318,JX2dXFlJi6A2-ZC3Cbzrk6OrAVex80DXcZUIv0oFVxaqlX...,25,Morgana,200,UTILITY,UTILITY,False


#### 8c. Exporting support picks to CSV for visualization

In [10]:
from pathlib import Path

if "support_picks_df" not in globals():
    raise RuntimeError("Run section 8b first so support_picks_df exists.")

csv_candidates = [
    Path("support_picks.csv"),
    Path("Week 5") / "support_picks.csv",
    Path("..") / "Week 5" / "support_picks.csv",
]
csv_path = next((p for p in csv_candidates if p.parent.exists()), csv_candidates[0])

support_picks_df.to_csv(csv_path, index=False)
print("Saved support picks CSV to:", csv_path.resolve())
print("rows exported:", len(support_picks_df))

Saved support picks CSV to: /Users/ruofuli/hcde530/Week 5/support_picks.csv
rows exported: 996
